# 06 · Versiones, time travel y mantenimiento

**Objetivo:** usar el historial que Delta fue creando mientras construíamos el pipeline.
Las pruebas destructivas (DELETE, RESTORE, VACUUM agresivo) se hacen sobre una **copia** (`silver_demo`), nunca sobre las tablas reales.

| Paso | Qué haces |
|---|---|
| 1 | Leer el historial de silver |
| 2 | Consultar el pasado: `VERSION AS OF` / `TIMESTAMP AS OF` |
| 3 | Un `DELETE` por error y `RESTORE` |
| 4 | `OPTIMIZE` + `ZORDER BY` sobre la tabla de hechos |
| 5 | `VACUUM` y su precio |
| 6 | Lo mismo desde PySpark: `DeltaTable` |

In [0]:
%run ./00_Setup

## Paso 1 · El historial de silver
Cada `MERGE` de `build_silver()` dejó una versión con sus métricas.

In [0]:
show_history(SILVER)

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {SILVER}").select("format", "numFiles", "sizeInBytes", "properties"))

👀 **Qué observar:** `operationMetrics` muestra filas insertadas, actualizadas y borradas en cada `MERGE`.
En `properties` aparece `delta.enableChangeDataFeed = true`.

## Paso 2 · Consultar el pasado
Cada versión es una foto completa de la tabla. Consultarla **no la modifica**.

In [0]:
for v in range(table_version(SILVER) + 1):
    filas = spark.sql(f"SELECT count(*) FROM {SILVER} VERSION AS OF {v}").first()[0]
    print(f"versión {v}: {filas} vuelos")

In [0]:
momento = spark.sql(f"DESCRIBE HISTORY {SILVER}").where("version = 1").first()["timestamp"]
filas = spark.sql(f"SELECT count(*) FROM {SILVER} TIMESTAMP AS OF '{momento}'").first()[0]
print(f"En {momento} silver tenía {filas} vuelos")

## Paso 3 · Un `DELETE` por error y `RESTORE`
Trabajamos sobre una copia: borramos "por error" todos los vuelos de un día y los recuperamos.

In [0]:
spark.sql(f"CREATE OR REPLACE TABLE silver_demo AS SELECT * FROM {SILVER}")

dia = spark.sql("SELECT max(fecha) FROM silver_demo").first()[0]
spark.sql(f"DELETE FROM silver_demo WHERE fecha = '{dia}'")

display(spark.sql("""
    SELECT (SELECT count(*) FROM silver_demo)                   AS vuelos_ahora,
           (SELECT count(*) FROM silver_demo VERSION AS OF 0)   AS vuelos_version_0
"""))
print("Día borrado por error:", dia)

In [0]:
spark.sql("RESTORE TABLE silver_demo TO VERSION AS OF 0")
print("Vuelos después del RESTORE:", spark.table("silver_demo").count())
show_history("silver_demo")

👀 **Qué observar:** el `DELETE` creó la versión 1; `RESTORE` creó la 2 y devolvió los vuelos.
El historial no se pierde: el propio `RESTORE` queda auditado.

## Paso 4 · `OPTIMIZE` + `ZORDER BY` sobre la tabla de hechos
Compacta archivos pequeños y agrupa las filas por las columnas más filtradas en Power BI (fecha y aerolínea).
Es seguro sobre la tabla real: no cambia los datos, solo cómo están guardados.

In [0]:
display(spark.sql(f"OPTIMIZE {FACT} ZORDER BY (fecha_key, aerolinea_icao)")
        .select("metrics.numFilesAdded", "metrics.numFilesRemoved"))

👀 **Qué observar:** `numFilesRemoved` / `numFilesAdded`. Con cada corrida diaria del Job se crean archivos pequeños;
`OPTIMIZE` los junta. Con tablas pequeñas el efecto es mínimo; con millones de filas, es la diferencia entre lento y rápido.

## Paso 5 · `VACUUM` y su precio
Los archivos de versiones viejas siguen ocupando espacio hasta que `VACUUM` los borra. Por defecto conserva **7 días**.
**Solo sobre la copia** bajamos la retención a 0 para ver qué se pierde.

In [0]:
display(spark.sql("VACUUM silver_demo DRY RUN"))

In [0]:
spark.sql("ALTER TABLE silver_demo SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
spark.sql("VACUUM silver_demo RETAIN 0 HOURS")

try:
    spark.sql("SELECT count(*) FROM silver_demo VERSION AS OF 1").first()
    print("La versión 1 todavía se puede leer")
except Exception as e:
    print("❌ La versión 1 ya no existe:", str(e).splitlines()[0][:160])

👀 **Qué observar:** tras `VACUUM` con retención 0, el time travel a versiones viejas deja de funcionar.

⚠️ **Nunca sobre silver ni bronze reales:** el CDF también lee versiones antiguas. Si `VACUUM` las borra antes de que
la siguiente capa las procese, el pipeline incremental se rompe. **En producción: 7 días o más.**

## Paso 6 · Lo mismo desde PySpark: `DeltaTable`

In [0]:
dt = DeltaTable.forName(spark, qualified_name(FACT))
display(dt.history().select("version", "timestamp", "operation").orderBy("version"))   # DESCRIBE HISTORY
print("Versión actual de fact:", dt.history().agg(F.max("version")).first()[0])

In [0]:
spark.sql("DROP TABLE IF EXISTS silver_demo")
print("Copia de prueba eliminada")

## ✔ Checkpoint
- [ ] Leí el historial y las métricas de cada `MERGE`
- [ ] Consulté versiones anteriores sin modificar la tabla
- [ ] Recuperé un borrado con `RESTORE`
- [ ] Optimicé la tabla de hechos con `ZORDER BY`
- [ ] Entiendo por qué `VACUUM` agresivo rompe el time travel y el CDF

**Siguiente:** `07_Unity_Catalog`